# 7장 차트 다듬기 II: 범례·배치·스타일과 AI 검수

『AI 시대의 파이썬 데이터 시각화』 예제 노트북입니다. 셀 첫 줄의 번호(P·C·B)는 책 지면의 코드 번호와 같습니다. 위에서부터 차례로 실행합니다.

## 준비

새 노트북이면 이 장의 코드보다 먼저 아래 준비 셀을 실행합니다(책의 앞 장과 같은 셀).

**코랩이면 이 셀(P02)을, 내 컴퓨터면 다음 셀(P03)을 실행합니다.** 둘 다 실행하지 않습니다.

In [ ]:
# P02 | 한글 글꼴(코랩): 나눔글꼴을 설치하고 Matplotlib에 등록합니다.
!apt-get -qq update
!apt-get -qq install -y fonts-nanum

import matplotlib.pyplot as plt
from matplotlib import font_manager

for path in font_manager.findSystemFonts():
    if 'Nanum' in path:
        font_manager.fontManager.addfont(path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

**내 컴퓨터(Mac·Windows)면 이 셀을 실행합니다.** Windows는 글꼴 이름을 `'Malgun Gothic'`으로 바꿉니다.

In [ ]:
# P03 | 한글 글꼴(Mac·Windows): 그래프를 그리기 전에 한 번 실행합니다.
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'AppleGothic'   # Windows는 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False    # 음수 기호가 깨지지 않게

## 7.1 질문과 검수 순서

In [ ]:
# C01 | 월별 표: 6장과 같은 표에서 검산할 합계와 월평균을 먼저 봅니다.
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

monthly = pd.DataFrame({
    'month': [1, 2, 3, 4] * 2,
    'branch': ['A'] * 4 + ['B'] * 4,
    'sales': [100, 120, 150, 130, 70, 90, 100, 120],   # 매출(백만원)
    'profit': [10, 15, 25, 20, 7, 9, 12, 18],          # 영업이익(백만원)
})
branch_colors = {'A': '#0072B2', 'B': '#D55E00'}   # 지점 색(4장과 같음)
a = monthly[monthly['branch'] == 'A']              # A지점 행만
print(monthly.groupby('branch')['sales'].agg(['sum', 'mean']))

## 7.2 범례·직접 라벨·격자

In [ ]:
# C02 | 바깥 범례: 범례를 Axes 오른쪽 밖에 붙여 선을 가리지 않게 합니다.
fig, ax = plt.subplots(figsize=(6, 2.4), layout='constrained')
for branch, part in monthly.groupby('branch'):
    ax.plot(part['month'], part['sales'], color=branch_colors[branch],
            linestyle='-' if branch == 'A' else '--',
            marker='o' if branch == 'A' else 's', label=branch)
ax.set(title='지점별 월 매출', xlabel='월', ylabel='매출 (백만원)',
       xticks=[1, 2, 3, 4], ylim=(0, 175))
ax.legend(title='지점', loc='upper left', bbox_to_anchor=(1.02, 1),
          frameon=False)                # 범례의 왼쪽 위를 (1.02, 1)에
plt.show()

In [ ]:
# C03 | 직접 라벨: 선 끝에 지점 이름과 값을 적어 범례를 대신합니다.
fig, ax = plt.subplots(figsize=(6, 2.4), layout='constrained')
for branch, part in monthly.groupby('branch'):
    ax.plot(part['month'], part['sales'], color=branch_colors[branch],
            linestyle='-' if branch == 'A' else '--',
            marker='o' if branch == 'A' else 's')
    last = part.iloc[-1]                          # 4월 행
    ax.annotate(f"{branch}: {last['sales']}",
                xy=(last['month'], last['sales']), va='center',
                xytext=(8, 4 if branch == 'A' else -4),   # 위아래로 비킴
                textcoords='offset points', color=branch_colors[branch])
ax.set(title='4월 매출: A 130, B 120백만원', xlabel='월', ylim=(0, 175),
       ylabel='매출 (백만원)', xticks=[1, 2, 3, 4], xlim=(0.8, 4.7))
plt.show()

In [ ]:
# C04 | 격자와 테두리: 진한 격자와 옅은 가로 격자를 나란히 봅니다.
fig, axes = plt.subplots(1, 2, figsize=(6, 2.2), layout='constrained',
                         sharey=True)
for ax in axes:
    ax.plot(a['month'], a['sales'], marker='o', color=branch_colors['A'])
    ax.set(xticks=[1, 2, 3, 4], ylim=(0, 180), xlabel='월')
axes[0].grid(True, color='#333333', linewidth=1.4)
axes[0].set(title='데이터와 경쟁하는 격자', ylabel='매출 (백만원)')
axes[1].grid(axis='y', color='#888888', alpha=0.25, linestyle='--')
axes[1].set_axisbelow(True)                 # 격자를 데이터 뒤로
axes[1].spines[['top', 'right']].set_visible(False)   # 위·오른쪽 테두리
axes[1].set_title('값 읽기를 돕는 격자')
plt.show()

### 직접 해 보기

**W7-01** Seaborn으로 그린 월별 매출 차트의 범례와 격자를 정리합니다.

1. `sns.lineplot()`에 `data=monthly`, `x='month'`, `y='sales'`, `hue='branch'`, `style='branch'`, `palette=branch_colors`, `errorbar=None`, `ax=ax`를 주어 4장처럼 그립니다.
2. C02처럼 범례를 Axes 오른쪽 바깥에 두고(`frameon=False`, 제목 "지점"), C04처럼 옅은 가로 격자만 데이터 뒤에 둡니다. 축 라벨과 단위도 답니다.
3. `ch07_output/W7-01.png`로 저장하고 다시 열어(2장 C05·C06) 범례나 글자가 잘리지 않았는지 봅니다.
4. 처음 그림과 견주어 달라진 점과 개선 이유 두 가지를 씁니다.

**완료 기준**: `bbox_to_anchor`로 범례를 바깥에 두었고, `set_axisbelow()`로 격자를 데이터 뒤에 두었으며, 저장본에 잘림이 없고, 개선 이유 두 가지가 있다.

In [ ]:
# W7-01 — 여기에서 직접 해 봅니다.

## 7.3 배치와 반환 객체

In [ ]:
# C05 | 배치: 두 지점을 같은 척도의 패널로 나누고 라벨은 한 번만 답니다.
fig, axes = plt.subplots(1, 2, figsize=(6, 2.4), layout='constrained',
                         sharey=True)
for ax, (branch, part) in zip(axes, monthly.groupby('branch')):
    ax.plot(part['month'], part['sales'], color=branch_colors[branch],
            linestyle='-' if branch == 'A' else '--',
            marker='o' if branch == 'A' else 's')
    ax.set(title=f'{branch}지점', xticks=[1, 2, 3, 4], ylim=(0, 180))
    ax.grid(axis='y', alpha=0.2)
fig.suptitle('같은 척도로 본 지점별 매출')
fig.supxlabel('월', fontsize=10)            # 두 패널 공통 X축 라벨
fig.supylabel('매출 (백만원)', fontsize=10)  # 공통 Y축 라벨
plt.show()

### 직접 해 보기

**W7-02** 지점별 패싯을 같은 기준으로 마무리합니다.

1. `sns.relplot()`에 `data=monthly`, `x='month'`, `y='sales'`, `col='branch'`, `hue='branch'`, `style='branch'`, `palette=branch_colors`, `kind='line'`, `marker='o'`, `errorbar=None`을 주어 두 패널을 그립니다.
2. `g.set()`으로 Y축 범위를 0~180, 눈금을 1~4월로 맞추고, `g.set_titles()`로 패널 제목을, `g.set_axis_labels()`로 축 라벨과 단위를 답니다.
3. `for ax in g.axes.flat:`으로 패널마다 옅은 가로 격자를 뒤에 둡니다.
4. 한 번에 건 설정과 패널마다 건 설정을 나눠 한 문장으로 씁니다.

**완료 기준**: `g.set()`·`g.set_titles()`·`g.axes.flat`을 썼고, 두 패널의 범위·눈금·단위가 같으며, 공통 설정과 패널별 설정을 구분한 문장이 있다.

In [ ]:
# W7-02 — 여기에서 직접 해 봅니다.

## 7.4 스타일 재사용과 AI 초안 검수

In [ ]:
# C06 | 스타일 재사용: 축 설정은 함수로, 전역 설정은 블록으로 묶습니다.
def style_sales(ax, title):
    # 월 매출(0~180백만원) 차트에 쓰는 축 설정입니다.
    ax.set(title=title, xlabel='월', ylabel='매출 (백만원)',
           xticks=[1, 2, 3, 4], ylim=(0, 180))
    ax.grid(axis='y', alpha=0.22, linestyle='--')
    ax.set_axisbelow(True)
    sns.despine(ax=ax)                      # 위·오른쪽 테두리를 한 번에

with plt.rc_context({'axes.titlesize': 13, 'lines.linewidth': 2.4}):
    fig, ax = plt.subplots(figsize=(6, 2.6), layout='constrained')
    ax.plot(a['month'], a['sales'], color=branch_colors['A'], marker='o')
    style_sales(ax, '재사용 스타일: A지점')
    plt.show()
print('블록 밖 제목 크기:', plt.rcParams['axes.titlesize'])

In [ ]:
# C07 | 스타일 시트: 옛 이름은 오류이고, whitegrid는 글꼴도 바꿉니다.
try:
    plt.style.use('seaborn')
except OSError as error:
    print('OSError:', str(error).split(',')[0])
print([name for name in plt.style.available if 'seaborn' in name][:3])
with plt.style.context('seaborn-v0_8-whitegrid'):
    print('블록 안 글꼴:', plt.rcParams['font.family'])

In [ ]:
# C08 | AI 초안(오류 예제): 검수 연습을 위해 직접 만든 초안입니다.
wrong = monthly.groupby('branch', as_index=False)['sales'].mean()
fig, ax = plt.subplots(figsize=(6, 2.4), layout='constrained')
bars = ax.bar(wrong['branch'], wrong['sales'],
              color=['#0072B2', '#D55E00'])
ax.bar_label(bars, padding=3)
ax.set(ylim=(80, 135), ylabel='매출', title='오류 예제: 1~4월 총매출')
print(wrong)
plt.show()

In [ ]:
# C09 | 초안 수정: 합계로 집계하고, 0에서 시작하는 축과 단위를 답니다.
totals = monthly.groupby('branch')['sales'].sum().reindex(['A', 'B'])
fig, ax = plt.subplots(figsize=(6, 2.4), layout='constrained')
bars = ax.bar(totals.index, totals.values,
              color=[branch_colors[k] for k in totals.index],
              edgecolor='black', linewidth=0)
bars[1].set_hatch('//')                      # B지점 빗금(4장)
ax.bar_label(bars, padding=3)
ax.set(ylim=(0, 580), xlabel='지점', ylabel='매출 (백만원)',
       title='1~4월 총매출: A 500, B 380백만원')
print('총매출:', totals.to_dict())
print('격차:', totals['A'] - totals['B'], '백만원')
plt.show()

In [ ]:
# C10 | 이중 Y축 점검: 오른쪽 축 범위만 바꿔도 교차가 달라집니다.
margin = a['profit'] / a['sales'] * 100      # A지점 영업이익률(%)
fig, axes = plt.subplots(1, 2, figsize=(6, 2.4), layout='constrained')
for ax, right in zip(axes, [(0, 40), (9, 17)]):
    ax.plot(a['month'], a['sales'], marker='o', color=branch_colors['A'])
    ax.set(xticks=[1, 2, 3, 4], ylim=(0, 180), xlabel='월')
    ax2 = ax.twinx()                         # 오른쪽에 두 번째 Y축
    ax2.plot(a['month'], margin, marker='^', linestyle=':',
             color=branch_colors['A'])       # A지점의 다른 지표(6장)
    ax2.set(ylim=right, title=f'오른쪽 축 {right[0]}~{right[1]}%')
axes[0].set(ylabel='매출 (백만원)')           # 바깥 두 축에만 라벨
ax2.set(ylabel='영업이익률 (%)')             # 마지막 ax2: 오른쪽 패널
plt.show()

In [ ]:
# C11 | W7-03 초안: 3~4월 총매출 검수 연습을 위해 만든 초안입니다.
period = monthly[monthly['month'].between(3, 4)]
draft = monthly.groupby('branch', as_index=False)['sales'].sum()
fig, ax = plt.subplots(figsize=(6, 2.4), layout='constrained')
bars = ax.bar(draft['branch'], draft['sales'],
              color=[branch_colors['B'], branch_colors['A']])
ax.bar_label(bars, padding=3)
ax.set(ylim=(0, 580), title='3~4월 지점별 총매출', ylabel='매출 (억원)')
print(draft)
plt.show()

### 직접 해 보기

**W7-03** C11은 "3~4월 지점별 총매출을 백만원 단위로 비교하고, A·B의 기존 색을 유지해 줘"라는 요청의 오류를 찾도록 만든 연습용 초안입니다.

1. 요청과 C11의 코드·출력·그림을 대조해 요청과 어긋나는 문제 세 가지와 그대로 둘 설정 하나를 근거와 함께 찾습니다.
2. 문제마다 문제·근거·수정·재확인을 한 행씩, 세 행으로 기록합니다.
3. 고친 코드와 별도로, 원자료에서 3월과 4월의 값을 더해 지점별 합계를 검산합니다.
4. 고친 그림을 `ch07_output/W7-03.png`로 저장하고 다시 엽니다.

**완료 기준**: 문제 세 가지를 네 칸으로 기록하고 그대로 둘 설정의 근거를 썼으며, 별도 검산이 고친 그림의 값과 같고 저장본을 확인했다.

In [ ]:
# W7-03 — 여기에서 직접 해 봅니다.

## 7.5 한 장 보고서와 동료 검수

In [ ]:
# C12 | 한 장 보고서: 넓은 추세 패널과 좁은 합계 패널을 함께 놓습니다.
totals = monthly.groupby('branch')['sales'].sum().reindex(['A', 'B'])
fig, axd = plt.subplot_mosaic([['trend', 'trend', 'total']],
                              figsize=(6, 2.3), layout='constrained')
for branch, part in monthly.groupby('branch'):
    axd['trend'].plot(part['month'], part['sales'],
                      color=branch_colors[branch],
                      linestyle='-' if branch == 'A' else '--',
                      marker='o' if branch == 'A' else 's')
    last = part.iloc[-1]                          # 4월 행에 값(C03)
    axd['trend'].annotate(f"{branch}: {last['sales']}", va='center',
                          xy=(last['month'], last['sales']),
                          xytext=(6, 4 if branch == 'A' else -4),
                          textcoords='offset points',
                          color=branch_colors[branch])
style_sales(axd['trend'], '월별 추세')
axd['trend'].set_xlim(0.8, 4.6)                  # 직접 라벨 자리(C03)
axd['trend'].text(0.02, 0.04, '자료: 교육용 가상 자료(월 합계)',
                  transform=axd['trend'].transAxes,
                  fontsize=9, color='#666666')           # 출처 줄(6장)
bars = axd['total'].bar(totals.index, totals.values,
                        color=[branch_colors[k] for k in totals.index],
                        edgecolor='black', linewidth=0)
bars[1].set_hatch('//')
axd['total'].bar_label(bars, padding=3)
axd['total'].set(title='1~4월 합계', xlabel='지점',
                 ylabel='매출 합계 (백만원)', ylim=(0, 580))
sns.despine(ax=axd['total'])
fig.suptitle('A지점의 총매출이 크지만 4월 격차는 10백만원', fontsize=13)
plt.show()

### 직접 해 보기

**W7-04** C12의 보고서를 영업이익으로 다시 만듭니다.

1. 왼쪽에 지점별 월 영업이익, 오른쪽에 1~4월 영업이익 합계를 그립니다. 지점 색과 B지점의 파선·빗금은 그대로 둡니다.
2. `style_sales()`는 월 매출(0~180백만원)에 맞춘 함수라 그대로 쓰지 않습니다. 영업이익에 맞는 범위와 라벨로 고친 함수를 왼쪽 패널에 쓰고, 오른쪽 패널의 범위와 라벨도 영업이익 합계에 맞춥니다.
3. 합계는 표에서 따로 계산해 출력합니다. 두 패널의 축은 따로 두고 라벨이 선을 가리면 옮기며, 전체 제목에 두 패널을 잇는 관찰을 적습니다.
4. `ch07_output/W7-04.png`로 저장해 다시 열고 해석 두 문장과 검수 기록(계산·척도·단위·범례·출처·저장본)을 씁니다.

**완료 기준**: 지점 색이 C12와 같고, 두 패널의 범위와 라벨이 영업이익에 맞으며, 합계를 따로 검산했고, 두 패널의 축이 독립이며, 해석 두 문장과 검수 기록이 있고 저장본을 확인했다.

In [ ]:
# W7-04 — 여기에서 직접 해 봅니다.

## 7.6 해석과 점검

## 직접 해 보기 풀이

먼저 위의 연습 칸에서 직접 해 본 뒤에 봅니다. 풀이는 한 가지 예이고, 완료 기준을 만족하면 다른 코드도 됩니다.

### W7-01

In [ ]:
# W7-01 | 풀이: Seaborn 월별 매출 차트의 범례와 격자를 정리합니다.
from pathlib import Path
from IPython.display import Image, display

output_dir = Path('ch07_output')
output_dir.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 2.4), layout='constrained')
sns.lineplot(data=monthly, x='month', y='sales', hue='branch', style='branch',
             palette=branch_colors, errorbar=None, ax=ax)            # 1. 4장처럼
ax.legend(title='지점', loc='upper left', bbox_to_anchor=(1.02, 1),
          frameon=False)                                             # 2. 범례는 오른쪽 바깥
ax.grid(axis='y', color='#888888', alpha=0.25, linestyle='--')       #    옅은 가로 격자
ax.set_axisbelow(True)                                               #    격자를 데이터 뒤로
ax.set(title='지점별 월 매출', xlabel='월', ylabel='매출 (백만원)',
       xticks=[1, 2, 3, 4], ylim=(0, 175))
fig.savefig(output_dir / 'W7-01.png', dpi=150, bbox_inches='tight')  # 3. 저장
plt.show()
display(Image(filename='ch07_output/W7-01.png', width=600))
# 개선 이유(예): 범례를 바깥으로 옮겨 선과 점을 가리지 않는다. 격자는 옅게 뒤로 두어 값 읽기만 돕고 선과 경쟁하지 않는다.

### W7-02

In [ ]:
# W7-02 | 풀이: 지점별 패싯을 같은 기준으로 마무리합니다.
g = sns.relplot(data=monthly, x='month', y='sales', col='branch', hue='branch',
                style='branch', palette=branch_colors, kind='line',
                marker='o', errorbar=None, height=2.4)
g.set(ylim=(0, 180), xticks=[1, 2, 3, 4])          # 2. 한 번에: 범위·눈금
g.set_titles('{col_name}지점')
g.set_axis_labels('월', '매출 (백만원)')
g.legend.set_title('지점')
for ax in g.axes.flat:                              # 3. 패널마다: 옅은 가로 격자
    ax.grid(axis='y', alpha=0.25, linestyle='--')
    ax.set_axisbelow(True)
plt.show()
assert all(ax.get_ylim() == (0, 180) for ax in g.axes.flat)
# 4.(예) 범위·눈금·패널 제목·축 라벨은 g.set·g.set_titles·g.set_axis_labels로 두 패널에 한 번에 걸었고, 격자는 g.axes.flat으로 패널마다 걸었다.

### W7-03

In [ ]:
# W7-03 | 풀이: C11의 문제 셋을 고치고 따로 검산합니다.
# | 문제 | 근거 | 수정 | 재확인 |
# | 기간이 1~4월 | period를 만들고도 monthly 전체를 합함(500·380) | period로 합계 | 별도 검산 280·220 |
# | 단위가 억원 | ylabel='매출 (억원)'인데 값은 백만원 | '매출 (백만원)' | 축 라벨과 막대 값 |
# | 색이 바뀜 | color=[B지점 색, A지점 색] | 지점 이름으로 사전에서 | A 파랑·B 주황 |
# 그대로 둔 설정(예): 제목 '3~4월 지점별 총매출'과 막대 값 라벨(bar_label) — 요청한 기간·비교와 맞고, 값을 그림에서 바로 대조하게 한다.
from pathlib import Path
from IPython.display import Image, display

period = monthly[monthly['month'].between(3, 4)]
fixed = period.groupby('branch', as_index=False)['sales'].sum()
output_dir = Path('ch07_output')
output_dir.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 2.4), layout='constrained')
bars = ax.bar(fixed['branch'], fixed['sales'],
              color=[branch_colors[b] for b in fixed['branch']])
ax.bar_label(bars, padding=3)
ax.set(ylim=(0, 320), title='3~4월 지점별 총매출', xlabel='지점',
       ylabel='매출 (백만원)')
fig.savefig(output_dir / 'W7-03.png', dpi=150, bbox_inches='tight')
plt.show()
display(Image(filename='ch07_output/W7-03.png', width=600))

In [ ]:
# 3. 원자료에서 3월과 4월의 값을 더해 검산
sales_a, sales_b = [100, 120, 150, 130], [70, 90, 100, 120]
check = {'A': sales_a[2] + sales_a[3], 'B': sales_b[2] + sales_b[3]}
print('검산:', check, '| 그림:', dict(zip(fixed['branch'], fixed['sales'])))
assert check == dict(zip(fixed['branch'], fixed['sales'])) == {'A': 280, 'B': 220}

### W7-04

In [ ]:
# W7-04 | 풀이: C12의 보고서를 영업이익으로 다시 만듭니다.
from pathlib import Path
from IPython.display import Image, display


def style_profit(ax, title):
    # 월 영업이익(0~30백만원) 차트에 쓰는 축 설정입니다.
    ax.set(title=title, xlabel='월', ylabel='영업이익 (백만원)',
           xticks=[1, 2, 3, 4], ylim=(0, 30))
    ax.grid(axis='y', alpha=0.22, linestyle='--')
    ax.set_axisbelow(True)
    sns.despine(ax=ax)


profit_totals = monthly.groupby('branch')['profit'].sum().reindex(['A', 'B'])
print(profit_totals)                                # 3. 합계를 표에서 따로 계산

output_dir = Path('ch07_output')
output_dir.mkdir(exist_ok=True)
fig, axd = plt.subplot_mosaic([['trend', 'trend', 'total']],
                              figsize=(6, 2.3), layout='constrained')
for branch, part in monthly.groupby('branch'):
    axd['trend'].plot(part['month'], part['profit'], color=branch_colors[branch],
                      linestyle='-' if branch == 'A' else '--',
                      marker='o' if branch == 'A' else 's')
    last = part.iloc[-1]                            # 4월 값을 선 끝에(C12와 같음)
    axd['trend'].annotate(f"{branch}: {last['profit']}", xy=(last['month'], last['profit']), va='center',
                          xytext=(6, 4 if branch == 'A' else -4), textcoords='offset points',
                          color=branch_colors[branch])
style_profit(axd['trend'], '월별 추세')
axd['trend'].set_xlim(0.8, 4.6)                     # 직접 라벨 자리(C12와 같음)
axd['trend'].text(0.98, 0.04, '자료: 교육용 가상 자료(월 합계)', ha='right',
                  transform=axd['trend'].transAxes, fontsize=9, color='#666666')
bars = axd['total'].bar(profit_totals.index, profit_totals.values,
                        color=[branch_colors[k] for k in profit_totals.index])
bars[1].set_hatch('//')
axd['total'].bar_label(bars, padding=3)
axd['total'].set(title='1~4월 합계', xlabel='지점',
                 ylabel='영업이익 합계 (백만원)', ylim=(0, 85))
sns.despine(ax=axd['total'])
fig.suptitle('A지점 영업이익 합계가 크지만 4월 격차는 2백만원', fontsize=13)
fig.savefig(output_dir / 'W7-04.png', dpi=150, bbox_inches='tight')
plt.show()
display(Image(filename='ch07_output/W7-04.png', width=600))
assert list(profit_totals) == [70, 46]
# 해석(예): 1~4월 영업이익 합계는 A지점 70, B지점 46백만원으로 A지점이 24백만원 크다.
#   그러나 4월만 보면 A지점 20, B지점 18백만원으로 차이가 2백만원까지 좁혀졌다.
# 검수 기록(예): 계산 — 합계를 표에서 따로 출력(70·46) / 척도 — 두 패널의 Y축 독립(0~30, 0~85) /
#   단위 — 축 라벨에 백만원 / 범례 — 선 끝 직접 라벨로 대신 / 출처 — 왼쪽 패널 아래 / 저장본 — 다시 열어 잘림 없음